In [2]:
from __future__ import annotations

import ast
import json
import os
from typing import Any

from google import genai
from pydantic import BaseModel, ConfigDict, Field

Two-pass Gemini SAS-to-Python translator with reflection.

Flow:
    Inputs -> Translate -> Reflect -> Improve -> Validate

Install:
    pip install google-genai pydantic pandas numpy

Environment:
    GEMINI_API_KEY

## Step 1: Structured Output Schema

In [3]:
class TranslationResult(BaseModel):
    """Machine-readable translation output."""

    model_config = ConfigDict(extra="forbid")

    code: str
    checkpoints: list[str]
    conflicts: list[dict[str, Any]]
    assumptions: list[str]
    helpers_requested: list[str]


class ReflectionResult(BaseModel):
    """Critique of the first translation."""

    model_config = ConfigDict(extra="forbid")

    overall_assessment: str
    issues: list[dict[str, Any]]
    missing_checkpoints: list[str]
    semantic_risks: list[str]
    required_changes: list[str]
    unresolved_questions: list[str]


## Step 2: Prompts

In [4]:
TRANSLATION_SYSTEM = """
You are a senior SAS-to-Python translation engineer.

Translate exactly one SAS unit into Python using pandas 3.x and NumPy.

Rules:
- Preserve SAS behavior, not merely the apparent business intent.
- Treat facts in the validated specification as immutable.
- Use the original SAS source to identify semantic discrepancies.
- Never silently resolve source/spec conflicts.
- One Python function per translated unit.
- Return a dictionary of named output DataFrames and any required
  macro-variable outputs.
- No file I/O inside translated functions.
- No printing or logging of data values.
- Only pandas, NumPy, and the Python standard library.
- Include SAS line references in code comments.
- Emit a checkpoint at every SAS dataset write, in source order.
- Preserve row ordering, column ordering, types, and missing-value behavior.
- Do not invent unavailable upstream functions or helper implementations.
- Request unavailable helpers explicitly.
- Do not claim execution or equivalence testing that has not occurred.

Return only the structured TranslationResult.
"""

REFLECTION_SYSTEM = """
You are the independent Reflection Agent reviewing a proposed
SAS-to-Python translation.

You must critique the translation; do not rewrite its code.

Compare the original SAS source, validated specification, project rules,
and initial translation.

Look for:
1. SAS semantics incorrectly translated into pandas.
2. Missing-value and type-conversion mismatches.
3. Incorrect aggregation, standardization, sorting, BY-group or merge behavior.
4. Missing, extra, reordered, or incorrectly placed checkpoints.
5. Output signature, column order, row order, and shape violations.
6. Unsupported assumptions or invented behavior.
7. Unapproved imports, file I/O, printing, or logging of data.
8. Contradictions between the SAS source and specification.
9. Potential runtime errors and unhandled edge cases.

Do not assume the initial translation is correct.
Do not invent SAS behavior when evidence is insufficient.
Separate definite defects from risks requiring source or runtime evidence.
Never modify immutable facts.

Return only the structured ReflectionResult.
"""

IMPROVEMENT_SYSTEM = """
You are the Translation Improvement Agent.

You receive the original translation inputs, the initial translation,
and an independent reflection report.

Produce a revised TranslationResult that addresses every actionable
reflection finding that can be resolved from available evidence.

Rules:
- Re-read the original SAS source and validated specification.
- The original SAS source and validated contract are essential evidence;
  the reflection is advice, not an authority.
- Do not blindly accept incorrect reflection findings.
- Preserve immutable facts.
- Do not invent behavior to eliminate a conflict.
- Record unresolved contradictions in conflicts.
- Record remaining uncertainty in assumptions or conflicts.
- Preserve all required checkpoints and source line comments.
- Follow the project's approved imports, function signature, and return
  conventions.
- Do not claim that code was executed or validated unless it was.
- Return the complete revised function, not a patch or diff.
- Return only the structured TranslationResult.
"""

## Step 3: Gemini Client

In [6]:
def make_client() -> genai.Client:
    """Initialize the Gemini client using GEMINI_API_KEY."""

    if not os.getenv("GEMINI_API_KEY"):
        raise RuntimeError(
            "GEMINI_API_KEY is not set in the environment."
        )

    return genai.Client(api_key=os.environ["GEMINI_API_KEY"])


def call_structured(
    client: genai.Client,
    model: str,
    system_instruction: str,
    payload: dict[str, Any],
    schema: type[BaseModel],
) -> BaseModel:
    """Make one Gemini call and parse its structured response."""

    response = client.models.generate_content(
        model=model,
        contents=json.dumps(payload, ensure_ascii=False),
        config={
            "system_instruction": system_instruction,
            "response_mime_type": "application/json",
            "response_schema": schema,
            "temperature": 0.1,
        },
    )

    if not response.text:
        raise RuntimeError("Gemini returned an empty response.")

    # Raises a validation exception if the response does not match schema.
    return schema.model_validate_json(response.text)

## Step 4: Deterministic Validation

In [7]:
REQUIRED_RESULT_KEYS = {
    "code",
    "checkpoints",
    "conflicts",
    "assumptions",
    "helpers_requested",
}

ALLOWED_IMPORT_ROOTS = {
    "pandas",
    "numpy",
    "math",
    "statistics",
    "decimal",
    "datetime",
    "collections",
    "itertools",
    "functools",
    "operator",
    "re",
    "typing",
    "dataclasses",
    "enum",
}


class TranslationPolicyError(ValueError):
    """Raised when generated code violates project policy."""


def validate_translation(
    result: TranslationResult,
) -> None:
    """
    Perform basic deterministic checks.

    This is a policy gate, not proof of semantic equivalence or a
    complete Python sandbox.
    """

    if set(result.model_dump()) != REQUIRED_RESULT_KEYS:
        raise TranslationPolicyError(
            "Translation result has unexpected or missing keys."
        )

    if not result.code.strip():
        raise TranslationPolicyError("Generated code is empty.")

    if len(set(result.checkpoints)) != len(result.checkpoints):
        raise TranslationPolicyError("Duplicate checkpoint names.")

    try:
        tree = ast.parse(result.code)
    except SyntaxError as exc:
        raise TranslationPolicyError(
            f"Generated code has invalid Python syntax: {exc}"
        ) from exc

    functions = [
        node for node in tree.body
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
    ]

    if len(functions) != 1 or not isinstance(
        functions[0], ast.FunctionDef
    ):
        raise TranslationPolicyError(
            "Expected exactly one top-level synchronous function."
        )

    for node in ast.walk(tree):
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            if isinstance(node, ast.Import):
                roots = [alias.name.split(".")[0] for alias in node.names]
            else:
                if node.module is None:
                    raise TranslationPolicyError(
                        "Relative imports are not allowed."
                    )
                roots = [node.module.split(".")[0]]

            forbidden = [
                root for root in roots
                if root not in ALLOWED_IMPORT_ROOTS
            ]

            if forbidden:
                raise TranslationPolicyError(
                    f"Unapproved imports: {forbidden}"
                )

        # Block common direct data-output mechanisms.
        if isinstance(node, ast.Call):
            if isinstance(node.func, ast.Name) and node.func.id in {
                "exec", "eval", "compile", "__import__",
                "open", "print", "breakpoint",
            }:
                raise TranslationPolicyError(
                    f"Disallowed function call: {node.func.id}"
                )

            if isinstance(node.func, ast.Attribute):
                if node.func.attr in {
                    "to_csv", "to_excel", "to_parquet",
                    "to_pickle", "to_json", "to_sql",
                    "tofile", "save", "dump",
                }:
                    raise TranslationPolicyError(
                        f"Potential file/output operation: "
                        f"{node.func.attr}"
                    )

    # The checkpoint callback is part of the required runtime interface.
    checkpoint_calls = [
        node for node in ast.walk(tree)
        if isinstance(node, ast.Call)
        and isinstance(node.func, ast.Name)
        and node.func.id == "checkpoint"
    ]

    if result.checkpoints and not checkpoint_calls:
        raise TranslationPolicyError(
            "Checkpoints are declared but no checkpoint calls were found."
        )

    if result.conflicts:
        # A conflict must be resolved by the workflow policy before
        # a translation is accepted as ready for execution.
        raise TranslationPolicyError(
            f"Unresolved translation conflicts: {len(result.conflicts)}"
        )

    if result.helpers_requested:
        raise TranslationPolicyError(
            "Translation requires unavailable helpers: "
            + ", ".join(result.helpers_requested)
        )


## Step 5: Full Three Call Workflow

In [8]:
def translate_with_reflection(
    *,
    spec: dict[str, Any],
    sas_source: str,
    behavior_notes: dict[str, Any],
    upstream_signatures: dict[str, Any],
    project_conventions: dict[str, Any],
    model: str = "gemini-2.5-pro",
    client: genai.Client | None = None,
    validate: bool = True,
) -> dict[str, Any]:
    """
    Run:
        1. Initial translation
        2. Independent reflection
        3. Improved translation

    Returns all stages for debugging, logging metadata, and auditing.
    Does not execute generated code.
    """

    client = client or make_client()

    # Keep the original inputs identical across the translation stages.
    original_inputs = {
        "spec": spec,
        "sas_source": sas_source,
        "behavior_notes": behavior_notes,
        "upstream_signatures": upstream_signatures,
        "project_conventions": project_conventions,
    }

    # ---- CALL 1: TRANSLATE ----
    initial = call_structured(
        client=client,
        model=model,
        system_instruction=TRANSLATION_SYSTEM,
        payload={
            **original_inputs,
            "task": "Translate this SAS unit into Python.",
        },
        schema=TranslationResult,
    )

    assert isinstance(initial, TranslationResult)

    # ---- CALL 2: REFLECT ----
    reflection = call_structured(
        client=client,
        model=model,
        system_instruction=REFLECTION_SYSTEM,
        payload={
            **original_inputs,
            "initial_translation": initial.model_dump(),
            "task": (
                "Independently audit the initial translation. "
                "Identify concrete defects and unresolved risks."
            ),
        },
        schema=ReflectionResult,
    )

    assert isinstance(reflection, ReflectionResult)

    # ---- CALL 3: IMPROVE ----
    improved = call_structured(
        client=client,
        model=model,
        system_instruction=IMPROVEMENT_SYSTEM,
        payload={
            **original_inputs,
            "initial_translation": initial.model_dump(),
            "reflection": reflection.model_dump(),
            "task": (
                "Produce the complete improved translation, "
                "resolving supported findings without inventing behavior."
            ),
        },
        schema=TranslationResult,
    )

    assert isinstance(improved, TranslationResult)

    validation_status = "not_run"
    validation_error = None

    if validate:
        try:
            validate_translation(improved)
            validation_status = "passed"
        except TranslationPolicyError as exc:
            validation_status = "failed"
            validation_error = str(exc)

    return {
        "initial_translation": initial.model_dump(),
        "reflection": reflection.model_dump(),
        "improved_translation": improved.model_dump(),
        "validation": {
            "status": validation_status,
            "error": validation_error,
            "execution_tested": False,
            "sas_equivalence_verified": False,
        },
    }



## Example

In [9]:
if __name__ == "__main__":
    # Replace these examples with the actual objects from your
    # orchestration layer. Do not use an incomplete spec in production.

    spec = {
        "unit": {
            "id": "M.grp_score",
            "kind": "macro",
            "source": "Star_Macros.sas",
            "lines": [28, 63],
            "source_hash": "replace-with-actual-source-hash",
        },
        "facts": {},
        "contract": {},
        "logic": [],
        "properties": [],
        "assumptions": [],
    }

    sas_source = """
    /* Insert the actual SAS source here. */
    """

    behavior_notes = {}
    upstream_signatures = {}
    project_conventions = {
        "pandas_version": "3.x",
        "allowed_imports": ["pandas", "numpy", "stdlib"],
        "checkpoint_call": "checkpoint(name, dataframe)",
        "no_file_io_inside_units": True,
        "no_logging_of_data_values": True,
    }

    result = translate_with_reflection(
        spec=spec,
        sas_source=sas_source,
        behavior_notes=behavior_notes,
        upstream_signatures=upstream_signatures,
        project_conventions=project_conventions,
    )

    # Print workflow metadata and diagnostics only, not translated
    # code or dataset contents.
    print(json.dumps({
        "validation": result["validation"],
        "reflection": result["reflection"],
        "checkpoints": result["improved_translation"]["checkpoints"],
    }, indent=2))

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


ValueError: additionalProperties is only supported in Gemini Enterprise Agent Platform mode, , not in Gemini Developer API mode.